In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = '../data'
IMG = '../app/images'
os.makedirs(IMG, exist_ok=True)

# Shared chart style for the website
C_DISM, C_ESC = '#8A9BA8', '#B5452B'   # dismissed / escalated
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 150, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.25,
                     'font.size': 11})

def save(name):
    plt.tight_layout()
    plt.savefig(f'{IMG}/{name}.png', bbox_inches='tight')
    plt.close()  # script mode: no windows, the chart is already saved to app/images
    print('Saved:', f'{IMG}/{name}.png')

## Load data

In [2]:
tr_sig = pd.read_csv(f'{DATA}/train_signals.csv', parse_dates=['signal_sanasi'])
te_sig = pd.read_csv(f'{DATA}/test_signals.csv', parse_dates=['signal_sanasi'])
tr_tx = pd.read_parquet(f'{DATA}/train_transactions.parquet')
tr_tx['tranzaksiya_vaqti'] = pd.to_datetime(tr_tx['tranzaksiya_vaqti'])

# Attach target and signal date to each transaction
tx = tr_tx.merge(tr_sig, on='signal_id', how='left')
tx['days_before'] = (tx['signal_sanasi'] + pd.Timedelta(days=1) - tx['tranzaksiya_vaqti']).dt.total_seconds() / 86400

print('Train alerts:', len(tr_sig), '| test alerts:', len(te_sig))
print('Train transactions:', len(tr_tx))
print('Median transactions per alert:', tr_tx.groupby('signal_id').size().median())
print(tx.head())

Train alerts: 14000 | test alerts: 6000
Train transactions: 6987663
Median transactions per alert: 461.0
   signal_id   tranzaksiya_vaqti kirim_chiqim tranzaksiya_turi  \
0  SG_016527 2024-07-06 08:47:18       chiqim   bank_otkazmasi   
1  SG_016527 2024-07-07 02:13:02       chiqim   bank_otkazmasi   
2  SG_016527 2024-07-08 11:15:12        kirim            karta   
3  SG_016527 2024-07-08 15:58:21        kirim            karta   
4  SG_016527 2024-07-09 02:33:17        kirim   bank_otkazmasi   

   miqdor_indeksi signal_sanasi  eskalatsiya  days_before  
0        0.236245    2025-01-02            0   180.633819  
1        1.130377    2025-01-02            0   179.907616  
2        0.466360    2025-01-02            0   178.531111  
3       -0.591761    2025-01-02            0   178.334479  
4       -1.108945    2025-01-02            0   177.893553  


## 1. Target distribution

In [3]:
counts = tr_sig['eskalatsiya'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Dismissed (0)', 'Escalated (1)'], counts.values, color=[C_DISM, C_ESC])
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width() / 2, v, f'{v:,}\n({v / counts.sum():.1%})', ha='center', va='bottom')
ax.set_title('Target distribution (train)'); ax.set_ylabel('Alerts')
save('01_target_distribution')
print(f"Escalation rate: {tr_sig['eskalatsiya'].mean():.2%}")

Saved: ../app/images/01_target_distribution.png
Escalation rate: 17.18%


**Insight:** There are more observations in the majority class (82.8%) compared to the minority class (17.18%). The imbalance problem is moderate and can be effectively addressed using stratified sampling techniques.

## 2. Alerts over time

In [4]:
fig, ax1 = plt.subplots(figsize=(10, 4))
w_tr = tr_sig.set_index('signal_sanasi').resample('W')['eskalatsiya'].agg(['size', 'mean'])
w_te = te_sig.set_index('signal_sanasi').resample('W').size()
ax1.plot(w_tr.index, w_tr['size'], label='Train: alerts per week', color='#2F5D7C')
ax1.plot(w_te.index, w_te.values, label='Test: alerts per week', color='#9C7A3C', linestyle='--')
ax1.set_ylabel('Alerts per week')
ax2 = ax1.twinx()
ax2.plot(w_tr.index, w_tr['mean'], color=C_ESC, alpha=0.7, label='Escalation rate (train)')
ax2.set_ylabel('Escalation rate'); ax2.grid(False)
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=9)
ax1.set_title('Weekly alerts: train vs test')
save('02_signals_over_time')
print('Train:', tr_sig.signal_sanasi.min().date(), '—', tr_sig.signal_sanasi.max().date())
print('Test: ', te_sig.signal_sanasi.min().date(), '—', te_sig.signal_sanasi.max().date())

Saved: ../app/images/02_signals_over_time.png
Train: 2025-01-01 — 2026-12-31
Test:  2025-01-01 — 2026-12-31


**Insight:** The train set and test set represent the same time frame: January 1, 2025, to December 31, 2026 (the time frames coincide exactly). This is confirmed by the adversarial validation results in `final.py` (AUC = 0.4987) — the value shows that train and test sets cannot be distinguished from each other, meaning random (time-agnostic) cross-validation can safely be used. The weekly escalation rate ranges from 6.8% to 28.4% (std ≈ 3.7 p.p.).

## 3. Transaction activity over time

In [5]:
m = tx.groupby(tx['tranzaksiya_vaqti'].dt.to_period('M')).size()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(m.index.to_timestamp(), m.values, color='#2F5D7C', marker='o', markersize=3)
ax.set_title('Transactions per month (train)'); ax.set_ylabel('Transactions')
save('03_tx_over_time')

Saved: ../app/images/03_tx_over_time.png


**Insight:** This graph has the form of a triangle: it increases from about 19k transactions/month (July 2024) to its maximum value of 388k (July/October 2025) and decreases to about 23k (December 2026). However, this has nothing to do with customer behavior — it is caused by the structure of the database. Each alert consists of a transaction history of about 180 days before its signal date, and the signal dates themselves are spread across 2025-2026.

## 4. Direction and transaction type

In [6]:
ct = pd.crosstab(tx['tranzaksiya_turi'], tx['kirim_chiqim'])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ct.plot(kind='bar', ax=axes[0], color=['#2F5D7C', '#9C7A3C'], rot=0)
axes[0].set_title('Transactions: type × direction'); axes[0].set_xlabel('')

share = (tx.groupby(['eskalatsiya', 'tranzaksiya_turi']).size()
           .groupby(level=0).transform(lambda s: s / s.sum()).unstack(0))
share.plot(kind='bar', ax=axes[1], color=[C_DISM, C_ESC], rot=0)
axes[1].set_title('Transaction type mix by alert outcome'); axes[1].set_xlabel('')
axes[1].legend(['Dismissed', 'Escalated'])
save('04_direction_type')
print(share.round(3))

Saved: ../app/images/04_direction_type.png
eskalatsiya           0      1
tranzaksiya_turi              
bank_otkazmasi    0.394  0.393
karta             0.538  0.537
naqd              0.063  0.065
xalqaro           0.005  0.005


**Insight:** The combination of transaction types is almost the same for escalated and dismissed cases: karta 53.8% vs 53.7%, bank_otkazmasi 39.4% vs 39.3%, naqd (cash) 6.29% vs 6.50%, xalqaro (international) 0.45% vs 0.47%. The difference is negligible; however, the pattern is as expected — the shares of naqd and xalqaro transactions are somewhat higher in escalated cases, which is a typical (if weak) AML signal.

## 5. Transaction size (`miqdor_indeksi`)

In [7]:
smp = tx.sample(min(len(tx), 500_000), random_state=42)
lo, hi = smp['miqdor_indeksi'].quantile([0.005, 0.995])
bins = np.linspace(lo, hi, 80)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for t, g in smp.groupby('tranzaksiya_turi'):
    axes[0].hist(g['miqdor_indeksi'], bins=bins, histtype='step', density=True, label=t, linewidth=1.5)
axes[0].set_title('By transaction type'); axes[0].legend(fontsize=9)
for y, col, lab in [(0, C_DISM, 'Dismissed'), (1, C_ESC, 'Escalated')]:
    axes[1].hist(smp.loc[smp.eskalatsiya == y, 'miqdor_indeksi'], bins=bins, histtype='step',
                 density=True, color=col, label=lab, linewidth=1.8)
axes[1].set_title('By alert outcome'); axes[1].legend()
for a in axes: a.set_xlabel('miqdor_indeksi')
save('05_amount_distribution')
print(tx.groupby('eskalatsiya')['miqdor_indeksi'].describe().round(3))

Saved: ../app/images/05_amount_distribution.png
                 count   mean    std    min    25%    50%    75%    max
eskalatsiya                                                            
0            5727996.0 -0.123  0.985 -2.912 -0.791 -0.235  0.442  6.692
1            1259667.0 -0.182  0.961 -2.912 -0.835 -0.285  0.376  6.464


**Insight:** Escalated alerts show slightly lower average `miqdor_indeksi` (mean −0.18, median −0.29) compared to dismissed alerts (mean −0.12, median −0.24) — escalation is not linked with "larger" amounts; if anything, the opposite. There is much more variance by transaction type: the "largest" type is xalqaro (mean 1.99), followed by naqd (0.53) and bank_otkazmasi (0.14), while karta is the smallest and negative (−0.43).

## 6. Activity before the alert
Average number of transactions per alert for each day before the alert date. The most important chart for modelling.

In [8]:
win = tx[(tx['days_before'] >= 0) & (tx['days_before'] < 90)].copy()
win['day'] = win['days_before'].astype(int)
n_sig = tr_sig['eskalatsiya'].value_counts()
act = win.groupby(['eskalatsiya', 'day']).size().unstack(0).fillna(0) / n_sig
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(act.index, act[0], color=C_DISM, label='Dismissed', linewidth=2)
ax.plot(act.index, act[1], color=C_ESC, label='Escalated', linewidth=2)
ax.invert_xaxis()
ax.set_xlabel('Days before alert'); ax.set_ylabel('Transactions per alert per day')
ax.set_title('Activity during the 90 days before the alert'); ax.legend()
save('06_activity_before_signal')
print('Share of last-7-day transactions within the 90-day window:')
print((win[win.day < 7].groupby('eskalatsiya').size() / win.groupby('eskalatsiya').size()).round(3))

Saved: ../app/images/06_activity_before_signal.png
Share of last-7-day transactions within the 90-day window:
eskalatsiya
0    0.212
1    0.207
dtype: float64


**Insight:** Background activity far from the signal (days 85-89) is steady at ~2.9-3.1 transactions/alert/day. On the day before the signal there is a sharp spike of ~40-42 transactions/alert (~14x the background rate), slightly higher for escalated cases (41.9 vs 40.3). Activity on the signal day itself (day=0) is very low, but escalated cases show three times as much (0.136 vs 0.044). Part of this spike may be a date-grouping artifact (`signal_sanasi` has no timestamp, so the whole last calendar day falls into one bin), but the elevated activity right before the signal is already captured by the window features (w1/w3/w7, etc.) in `final.py`.

## 7. Escalated vs dismissed alerts

In [9]:
g = tx.groupby('signal_id')
feat = pd.DataFrame({
    'Transactions': g.size(),
    'Mean miqdor_indeksi': g['miqdor_indeksi'].mean(),
    'Max miqdor_indeksi': g['miqdor_indeksi'].max(),
    'Cash (naqd) share': g['tranzaksiya_turi'].apply(lambda s: (s == 'naqd').mean()),
    'International share': g['tranzaksiya_turi'].apply(lambda s: (s == 'xalqaro').mean()),
    'Outgoing share': g['kirim_chiqim'].apply(lambda s: (s == 'chiqim').mean()),
    'Transactions, last 7 days': tx[tx.days_before.between(0, 7)].groupby('signal_id').size(),
    'Days since last transaction': g['days_before'].min(),
}).fillna({'Transactions, last 7 days': 0})
feat = feat.join(tr_sig.set_index('signal_id')['eskalatsiya'])

cols = [c for c in feat.columns if c != 'eskalatsiya']
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for ax, c in zip(axes.ravel(), cols):
    data = [feat.loc[feat.eskalatsiya == k, c].dropna() for k in (0, 1)]
    bp = ax.boxplot(data, showfliers=False, patch_artist=True, medianprops={'color': '#1C2630'})
    ax.set_xticks([1, 2]); ax.set_xticklabels(['Dismissed', 'Escalated'])
    for patch, col in zip(bp['boxes'], [C_DISM, C_ESC]): patch.set_facecolor(col); patch.set_alpha(0.6)
    ax.set_title(c, fontsize=10)
fig.suptitle('Escalated vs dismissed alerts', fontsize=13)
save('07_class_comparison')
print(feat.groupby('eskalatsiya').median().T.round(3))

Saved: ../app/images/07_class_comparison.png
eskalatsiya                        0        1
Transactions                 455.000  493.000
Mean miqdor_indeksi           -0.070   -0.144
Max miqdor_indeksi             2.992    2.817
Cash (naqd) share              0.059    0.062
International share            0.003    0.004
Outgoing share                 0.251    0.256
Transactions, last 7 days     36.000   37.000
Days since last transaction    1.000    1.000


**Insight:** For simple aggregates, the class differences are quite small: slightly more transactions (523.8 vs 494.0 on average), slightly higher recent (7-day) activity (47.3 vs 45.5), slightly higher shares of cash/international/outgoing transactions, and a slightly lower mean transaction value. No single feature separates the classes on its own. This is consistent with `final.py`, where the best OOF AUC on simple aggregates is only about 0.63, and advanced features (time-decay, pass-through patterns, type entropy) are needed to push it above 0.64.

## 8. Hour of day and day of week

In [10]:
tx['hour'] = tx['tranzaksiya_vaqti'].dt.hour
tx['dow'] = tx['tranzaksiya_vaqti'].dt.dayofweek
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for y, col, lab in [(0, C_DISM, 'Dismissed'), (1, C_ESC, 'Escalated')]:
    s = tx[tx.eskalatsiya == y]
    axes[0].plot(s['hour'].value_counts(normalize=True).sort_index(), color=col, label=lab, linewidth=2)
    axes[1].plot(s['dow'].value_counts(normalize=True).sort_index(), color=col, label=lab, linewidth=2, marker='o')
axes[0].set_title('Share of transactions by hour'); axes[0].set_xlabel('Hour')
axes[1].set_title('Share of transactions by weekday'); axes[1].set_xticks(range(7))
axes[1].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
for a in axes: a.legend()
save('08_hour_weekday')

Saved: ../app/images/08_hour_weekday.png


**Insight:** Timestamps are not always midnight, so time-of-day is real. Activity is otherwise evenly spread across weekdays (14.2-14.4% per day in both groups), with no meaningful weekend effect (weekend transactions are 28.5% in both groups) and no difference in night-time activity (00:00-05:00, ~23.1-23.2% in both groups).

**🚩 Anomaly:** transactions at hour 23:00 occur about **three times more often** (11.7% of all transactions) than at any other hour (~3.8% each) — clearly not a random distribution. It looks like a technical/data-quality artifact (e.g. records without an exact timestamp defaulting to end-of-day) rather than a behavioral pattern, and is flagged here as a data-quality issue for the team.

## Key findings for the website

1. The target is imbalanced, with only 17.2% escalated cases, so we use stratified cross-validation and PR-AUC as an additional evaluation metric.
2. Train and test sets cover an identical period (2025-01-01 to 2026-12-31). Adversarial validation shows the samples cannot be distinguished (AUC ≈ 0.50), so standard random cross-validation can safely be used.
3. Simple aggregated features computed over the whole period (transaction count, type shares, average amount) show very little difference between escalated and dismissed alerts — the signal is weak and spread across many features.
4. A sharp one-day spike in activity right before the signal (~14x the background rate) is the most informative pattern, and it is somewhat stronger for escalated alerts — this motivated the window and time-decay features in the model.
5. Indeed, the classical AML features (the proportion of cash and international operations) are slightly greater for cases with escalation, but the impact is very small by itself.
6. A technical anomaly has been found: 23:00 hour has 3 times more transactions than any other hour – this is probably the default time for those transactions for which the exact time was not specified.
7. The final model (ensemble of LightGBM, CatBoost and XGBoost with 80 features) provides OOF AUC of 0.6425; the problem is quite challenging, with the simplest features providing a random score and feature engineering giving the main contribution to the result.